# Notebook 04 · Clasificación multiclase con TensorFlow + Keras (Softmax)

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** clasificar frutas en **tres** categorías: 0 = manzana, 1 = banano, 2 = naranja.
El problema deja de ser binario y pasa a ser **multiclase**.

> **Alcance:** transición conceptual de binario a multiclase, función **Softmax**, pérdida
> `sparse_categorical_crossentropy`, arquitectura 3 → 6 → 3, evaluación con matriz de confusión y
> reporte de clasificación, e interpretación de las probabilidades por clase.

---

## 1. Diferencias entre clasificación binaria y multiclase

| Aspecto | Binaria (Notebook 03) | Multiclase (este notebook) |
|---|---|---|
| Número de clases | 2 | 3 o más (en MNIST serán 10) |
| Neuronas de salida | **1** | **una por clase** |
| Activación de salida | Sigmoide | **Softmax** |
| Interpretación | P(clase 1) | Vector de probabilidades que suma 1 |
| Pérdida | `binary_crossentropy` | `sparse_categorical_crossentropy` (etiquetas enteras) o `categorical_crossentropy` (etiquetas *one-hot*) |
| Decisión | `p >= 0.5` | `np.argmax(p)` |
| Métrica adecuada | *accuracy*, precisión, recall | *accuracy* + **matriz de confusión** (revela qué clases se confunden entre sí) |

## 2. Función Softmax

Convierte las salidas crudas (*logits*) $z_1,\dots,z_K$ de $K$ neuronas en una distribución de probabilidad:

$$ \mathrm{softmax}(z_i)=\frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
\qquad\text{con}\qquad
\sum_i \mathrm{softmax}(z_i)=1 $$

La clase predicha es la de mayor probabilidad, obtenida con `np.argmax`. La **entropía cruzada** mide
cuánto se aleja la distribución predicha de la verdadera (one-hot):

$$ L=-\frac{1}{m}\sum_{k}\log \hat y^{(k)}_{y^{(k)}} $$


## 3. Entorno y dataset multiclase

Se agregan 10 naranjas al dataset de los notebooks anteriores. Las naranjas son **más pesadas que las
manzanas** pero **cortas y poco amarillas**, de modo que no basta con un umbral sobre una única
característica: el modelo debe combinar las tres.

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import random

random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

I0000 00:00:1791644432.956849   25710 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791644434.533555   25710 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1


In [2]:
X_multi = np.array([
    # MANZANAS - clase 0
    [150, 7.5, .10], [160, 8.0, .15], [145, 7.2, .05], [170, 8.3, .20], [155, 7.8, .10],
    [165, 8.1, .15], [140, 7.0, .05], [175, 8.5, .20], [158, 7.9, .10], [162, 8.2, .15],

    # BANANOS - clase 1
    [120, 18.0, .90], [125, 19.0, .95], [115, 17.5, .85], [130, 20.0, 1.00], [118, 18.5, .90],
    [123, 19.5, .95], [110, 17.0, .80], [128, 20.5, 1.00], [121, 18.8, .90], [126, 19.2, .95],

    # NARANJAS - clase 2
    [180, 8.0, .35], [190, 8.5, .40], [175, 7.8, .30], [200, 9.0, .45], [185, 8.2, .35],
    [195, 8.7, .40], [170, 7.5, .30], [205, 9.2, .45], [188, 8.4, .35], [192, 8.6, .40]
], dtype=float)

y_multi = np.array([0] * 10 + [1] * 10 + [2] * 10)
nombres_frutas = ["MANZANA", "BANANO", "NARANJA"]

print("X_multi:", X_multi.shape, "-> (30 frutas, 3 características)")
print("y_multi:", y_multi.shape, "-> clases:", np.unique(y_multi))
print("Ejemplos por clase:", np.bincount(y_multi))
print()
for c, n in enumerate(nombres_frutas):
    m = y_multi == c
    print(f"{n:<9} peso {X_multi[m,0].mean():6.1f} g | longitud {X_multi[m,1].mean():5.2f} cm | amarillo {X_multi[m,2].mean():.2f}")

X_multi: (30, 3) -> (30 frutas, 3 características)
y_multi: (30,) -> clases: [0 1 2]
Ejemplos por clase: [10 10 10]

MANZANA   peso  158.0 g | longitud  7.85 cm | amarillo 0.12
BANANO    peso  121.6 g | longitud 18.80 cm | amarillo 0.92
NARANJA   peso  188.0 g | longitud  8.39 cm | amarillo 0.38


In [3]:
# División estratificada 80/20 y escalado
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

scaler_multi = StandardScaler()
X_train_multi = scaler_multi.fit_transform(X_train_multi)
X_test_multi = scaler_multi.transform(X_test_multi)

print("X_train_multi:", X_train_multi.shape, "| X_test_multi:", X_test_multi.shape)
print("Clases en entrenamiento:", np.bincount(y_train_multi))
print("Clases en prueba       :", np.bincount(y_test_multi))
print("Etiquetas de prueba    :", y_test_multi)

X_train_multi: (24, 3) | X_test_multi: (6, 3)
Clases en entrenamiento: [8 8 8]
Clases en prueba       : [2 2 2]
Etiquetas de prueba    : [2 1 2 0 1 0]


In [4]:
# Verificación del comportamiento de Softmax
z_ejemplo = np.array([[2.0, 1.0, 0.1]])
exp = np.exp(z_ejemplo)
softmax_manual = exp / exp.sum()

print("Logits z            :", z_ejemplo[0])
print("Softmax manual      :", np.round(softmax_manual[0], 4), "| suma:", round(softmax_manual.sum(), 6))
print("Clase elegida argmax:", int(np.argmax(softmax_manual)))
print()
z_otro = np.array([[0.4, 2.8, 1.1]])
s_otro = np.exp(z_otro) / np.exp(z_otro).sum()
print("Logits z            :", z_otro[0])
print("Softmax             :", np.round(s_otro[0], 4), "| argmax:", int(np.argmax(s_otro)))
print("-> Softmax conserva el orden de los logits y los normaliza para que sumen 1.")

Logits z            : [2.  1.  0.1]
Softmax manual      : [0.659  0.2424 0.0986] | suma: 1.0
Clase elegida argmax: 0

Logits z            : [0.4 2.8 1.1]
Softmax             : [0.0712 0.7853 0.1435] | argmax: 1
-> Softmax conserva el orden de los logits y los normaliza para que sumen 1.


## 4. Modelo 3 → 6 → 3

La capa oculta se amplía a **6 neuronas** porque ahora debe representar tres regiones del espacio de
características. La capa de salida tiene **3 neuronas con Softmax**, una por clase.

**Por qué `sparse_categorical_crossentropy`:** las etiquetas están como enteros (0, 1, 2). Si se
usaran vectores *one-hot* (`[1,0,0]`), la pérdida correcta sería `categorical_crossentropy`. Usar la
combinación equivocada produce errores de forma o entrenamientos que no convergen.

In [5]:
tf.random.set_seed(42)

modelo_multi = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(6, activation="relu", name="oculta"),
    keras.layers.Dense(3, activation="softmax", name="salida_softmax")
])

modelo_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_multi.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ oculta (Dense)                  │ (None, 6)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida_softmax (Dense)          │ (None, 3)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45 (180.00 B)

 Trainable params: 45 (180.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Entrenamiento

In [6]:
historial_multi = modelo_multi.fit(
    X_train_multi,
    y_train_multi,
    epochs=300,
    batch_size=4,
    validation_split=0.2,
    verbose=0
)

print("Error inicial:", round(historial_multi.history["loss"][0], 6))
print("Error final  :", round(historial_multi.history["loss"][-1], 6))
print()
print(f"{'Época':>7}{'Loss':>11}{'Acc':>8}{'Val loss':>11}{'Val acc':>9}")
for e in [0, 49, 99, 149, 199, 249, 299]:
    h = historial_multi.history
    print(f"{e+1:>7}{h['loss'][e]:>11.5f}{h['accuracy'][e]:>8.3f}"
          f"{h['val_loss'][e]:>11.5f}{h['val_accuracy'][e]:>9.3f}")

Error inicial: 1.469859
Error final  : 0.005629

  Época       Loss     Acc   Val loss  Val acc
      1    1.46986   0.000    1.30948    0.000
     50    0.10030   0.947    0.00284    1.000
    100    0.05633   1.000    0.00054    1.000
    150    0.03043   1.000    0.00021    1.000
    200    0.01625   1.000    0.00011    1.000
    250    0.00920   1.000    0.00007    1.000
    300    0.00563   1.000    0.00005    1.000


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
h = historial_multi.history

axes[0].plot(h["loss"], label="Entrenamiento", color="#c0392b", lw=2)
axes[0].plot(h["val_loss"], label="Validación", color="#2980b9", lw=1.6, ls="--")
axes[0].set_yscale("log")
axes[0].set_title("Figura 9. Pérdida multiclase")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("sparse_categorical_crossentropy")
axes[0].legend(fontsize=8)

axes[1].plot(h["accuracy"], label="Entrenamiento", color="#27ae60", lw=2)
axes[1].plot(h["val_accuracy"], label="Validación", color="#8e44ad", lw=1.6, ls="--")
axes[1].set_ylim(0.2, 1.05)
axes[1].set_title("Figura 10. Accuracy multiclase")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.savefig("fig06_keras_multiclase_curvas.png", bbox_inches="tight"); plt.show()

## 6. Evaluación y probabilidades por clase

In [8]:
loss_multi, accuracy_multi = modelo_multi.evaluate(X_test_multi, y_test_multi, verbose=0)

probabilidades_multi = modelo_multi.predict(X_test_multi, verbose=0)
predicciones_multi = np.argmax(probabilidades_multi, axis=1)
acc_manual = accuracy_score(y_test_multi, predicciones_multi)

print("=== CLASIFICACIÓN MULTICLASE (manzana / banano / naranja) ===")
print("Loss en prueba  :", round(float(loss_multi), 6))
print("Accuracy (Keras):", round(float(accuracy_multi), 4))
print("Accuracy (sklearn, verificación):", round(acc_manual, 4))
print()
print(f"{'#':>3} {'P(manzana)':>11} {'P(banano)':>10} {'P(naranja)':>11}   {'Predicción':>10} {'Real':>10} {'Resultado':>10}")
for i in range(len(X_test_multi)):
    p = probabilidades_multi[i]
    pred, real = nombres_frutas[predicciones_multi[i]], nombres_frutas[y_test_multi[i]]
    ok = "ACIERTO" if pred == real else "ERROR"
    print(f"{i+1:>3} {p[0]:>11.4f} {p[1]:>10.4f} {p[2]:>11.4f}   {pred:>10} {real:>10} {ok:>10}")
print()
print("Cada fila de probabilidades suma 1:",
      bool(np.allclose(probabilidades_multi.sum(axis=1), 1.0)))

=== CLASIFICACIÓN MULTICLASE (manzana / banano / naranja) ===
Loss en prueba  : 0.003006
Accuracy (Keras): 1.0
Accuracy (sklearn, verificación): 1.0

  #  P(manzana)  P(banano)  P(naranja)   Predicción       Real  Resultado
  1      0.0001     0.0000      0.9999      NARANJA    NARANJA    ACIERTO
  2      0.0001     0.9999      0.0000       BANANO     BANANO    ACIERTO
  3      0.0000     0.0000      1.0000      NARANJA    NARANJA    ACIERTO
  4      1.0000     0.0000      0.0000      MANZANA    MANZANA    ACIERTO
  5      0.0001     0.9999      0.0000       BANANO     BANANO    ACIERTO
  6      0.9825     0.0006      0.0169      MANZANA    MANZANA    ACIERTO

Cada fila de probabilidades suma 1: True


## 7. Matriz de confusión y reporte de clasificación

El *accuracy* global no dice **qué** clases se confunden. La matriz de confusión es la herramienta
adecuada en problemas multiclase y será la métrica principal en MNIST (10 clases).

In [9]:
cm = confusion_matrix(y_test_multi, predicciones_multi)
print("Matriz de confusión (filas = reales, columnas = predichas):")
print(cm)
print()
print(classification_report(y_test_multi, predicciones_multi,
                           target_names=nombres_frutas, zero_division=0))

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ConfusionMatrixDisplay(cm, display_labels=nombres_frutas).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Figura 11. Matriz de confusión - frutas multiclase")
plt.tight_layout(); plt.savefig("fig07_confusion_multiclase.png", bbox_inches="tight"); plt.show()

Matriz de confusión (filas = reales, columnas = predichas):
[[2 0 0]
 [0 2 0]
 [0 0 2]]

              precision    recall  f1-score   support

     MANZANA       1.00      1.00      1.00         2
      BANANO       1.00      1.00      1.00         2
     NARANJA       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



## 8. Frontera de decisión multiclase

Con tres clases, la superficie de decisión se compone de **tres regiones**. Se colorea cada punto de la
malla con la clase de mayor probabilidad.

In [10]:
media, desv = scaler_multi.mean_, scaler_multi.scale_
i_f, j_f = 1, 2   # longitud vs nivel de amarillo

g1 = np.linspace(X_multi[:, i_f].min() - 1.0, X_multi[:, i_f].max() + 1.0, 320)
g2 = np.linspace(X_multi[:, j_f].min() - 0.10, X_multi[:, j_f].max() + 0.10, 320)
G1, G2 = np.meshgrid(g1, g2)
pts = np.column_stack([np.full(G1.size, media[0]), G1.ravel(), G2.ravel()])
pts_e = (pts - media) / desv

probs_grid = modelo_multi.predict(pts_e, verbose=0)
clase_grid = np.argmax(probs_grid, axis=1).reshape(G1.shape)
conf_grid = probs_grid.max(axis=1).reshape(G1.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
colores_clase = ["#c0392b", "#f1c40f", "#e67e22"]
import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(["#f5b7b1", "#fcf3cf", "#fad7a0"])

axes[0].contourf(G1, G2, clase_grid, levels=[-0.5, 0.5, 1.5, 2.5], cmap=cmap, alpha=0.9)
for c, n in enumerate(nombres_frutas):
    m = y_multi == c
    axes[0].scatter(X_multi[m, i_f], X_multi[m, j_f], c=colores_clase[c],
                    edgecolors="k", s=70, label=n)
axes[0].set_xlabel("Longitud (cm)"); axes[0].set_ylabel("Nivel de amarillo")
axes[0].set_title("Figura 12. Regiones de decisión (Softmax + argmax)")
axes[0].legend(fontsize=8)

im = axes[1].pcolormesh(G1, G2, conf_grid, cmap="viridis", vmin=0.33, vmax=1.0, shading="auto")
for c, n in enumerate(nombres_frutas):
    m = y_multi == c
    axes[1].scatter(X_multi[m, i_f], X_multi[m, j_f], c=colores_clase[c],
                    edgecolors="w", s=70, label=n)
axes[1].set_xlabel("Longitud (cm)"); axes[1].set_ylabel("Nivel de amarillo")
axes[1].set_title("Figura 13. Confianza de la predicción (probabilidad máxima)")
axes[1].legend(fontsize=8, loc="lower right")
fig.colorbar(im, ax=axes[1], label="P(clase predicha)")

plt.tight_layout(); plt.savefig("fig08_frontera_multiclase.png", bbox_inches="tight"); plt.show()

print("Probabilidad máxima promedio en la malla:", round(float(conf_grid.mean()), 4))
print("Zonas con confianza < 0.60 (fronteras difusas):",
      f"{(conf_grid < 0.60).mean()*100:.2f} % del área")

Probabilidad máxima promedio en la malla: 0.936
Zonas con confianza < 0.60 (fronteras difusas): 4.20 % del área


## 9. Predicción de una fruta nueva

In [11]:
fruta_nueva_multi = np.array([[190.0, 8.5, 0.38]])
fruta_escalada_multi = scaler_multi.transform(fruta_nueva_multi)

prob = modelo_multi.predict(fruta_escalada_multi, verbose=0)
clase = int(np.argmax(prob, axis=1)[0])

print("Fruta nueva (peso, longitud, amarillo):", fruta_nueva_multi[0])
print("Probabilidades por clase:")
for n, p in zip(nombres_frutas, prob[0]):
    barra = "█" * int(round(p * 30))
    print(f"   {n:<9} {p:.4f}  {barra}")
print()
print("Clase predicha (argmax):", clase)
print("La red piensa que es     :", nombres_frutas[clase])

Fruta nueva (peso, longitud, amarillo): [190.     8.5    0.38]
Probabilidades por clase:
   MANZANA   0.0000  
   BANANO    0.0000  
   NARANJA   1.0000  ██████████████████████████████

Clase predicha (argmax): 2
La red piensa que es     : NARANJA


## 10. Análisis de resultados

- La pérdida multiclase parte de un valor cercano a $-\log(1/3)=1.0986$, que corresponde a predecir la
  misma probabilidad para las tres clases; su descenso confirma que el modelo está aprendiendo a
  discriminar.
- El *accuracy* sobre el conjunto de prueba debe leerse con cautela: son **6 frutas**, por lo que cada
  acierto representa ≈16.7 puntos porcentuales. La matriz de confusión aporta más información que el
  valor global.
- La **capa oculta de 6 neuronas ReLU** permite fronteras no necesariamente paralelas; con Softmax en la
  salida las regiones son convexas por construcción (el argmax de funciones lineales genera
  politopos), lo que es suficiente para este dataset.
- Este mismo esquema —`Flatten` → `Dense(ReLU)` → `Dense(softmax, n_clases)`— es exactamente el que se
  aplicará a MNIST con **10** clases en el Notebook 05, sustituyendo 3 características por 784 píxeles.

## 11. Conclusiones del notebook

1. Pasar de binario a multiclase implica tres cambios coordinados: **número de neuronas de salida**,
   **activación Softmax** y **función de pérdida de entropía cruzada multiclase**.
2. `argmax` selecciona la posición del valor mayor; esa posición **es** la clase predicha.
3. En problemas multiclase la evaluación debe incluir la matriz de confusión y el reporte por clase
   (precisión y recall), no únicamente el *accuracy*.
4. La confianza de la predicción (probabilidad máxima) permite identificar zonas ambiguas donde el
   modelo debería abstenerse o requerir revisión humana.

## 12. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
  https://github.com/PacktPublishing/Deep-Learning-with-TensorFlow-and-Keras-3rd-edition
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/